# Loading openff-pablo's prepared PDB corpus

Every file in Pablo's `prepared_pdbs` test set, loaded with `Protein`.
Templates outside the 34 shipped components download from the RCSB.

In [ ]:
import logging, warnings
from rdkit import RDLogger

warnings.filterwarnings("ignore")
logging.disable(logging.WARNING)
RDLogger.DisableLog("rdApp.*")

In [ ]:
# Pablo's own test corpus of prepared PDB files, fetched into a git-ignored cache.
import urllib.request
from pathlib import Path

CORPUS_URL = "https://raw.githubusercontent.com/openforcefield/openff-pablo/main/openff/pablo/_tests/data/prepared_pdbs/"
CORPUS_FILES = [
    "193l_prepared.pdb",
    "1FLR_prepared.pdb",
    "1a4t_samechain.pdb",
    "1csa_maestro.pdb",
    "1csa_maestro_waterfirst.pdb",
    "1hje_diffchain.pdb",
    "1hje_samechain.pdb",
    "1p3q_noter.pdb",
    "2MUM_blowup.pdb",
    "2MUM_composed_function.pdb",
    "2MUM_discontiguous_resseq.pdb",
    "2MUM_discontiguous_serial.pdb",
    "2MUM_dryrun.pdb",
    "2MUM_icode.pdb",
    "2MUM_letters_in_resseq.pdb",
    "2MUM_letters_in_serial.pdb",
    "2MUM_neutralized.pdb",
    "2MUM_reuse_resseq.pdb",
    "2MUM_reuse_serial.pdb",
    "2hi7_prepared.pdb",
    "2zuq_prepared.pdb",
    "3h34_prepared.pdb",
    "3ip9_dye_solvated.pdb",
    "5eil_fixed.pdb",
    "ions.pdb",
    "polyglycines.pdb"
]
cache = Path("../assets_cache/pablo_prepared_pdbs")
cache.mkdir(parents=True, exist_ok=True)
for name in CORPUS_FILES:
    if not (cache / name).exists():
        urllib.request.urlretrieve(CORPUS_URL + name, cache / name)
len(list(cache.glob("*.pdb"))), "files"

In [ ]:
from mbuild.biopolymers import Protein

loaded, failed = [], []
for path in sorted(cache.glob("*.pdb")):
    try:
        protein = Protein(path, download=True)
    except Exception as error:
        failed.append((path.name, str(error).splitlines()[0]))
        continue
    loaded.append(path.name)
    print(
        f"{path.name:32s} {len(protein.chains):3d} chains"
        f" {len(list(protein.residues())):5d} residues"
        f" {protein.n_particles:6d} atoms"
        f"  net charge {protein.net_formal_charge:+d}"
        f"  crosslinks {len(protein.bond_records())}"
    )
print()
print(len(loaded), "loaded,", len(failed), "refused")

Each refusal names the residue and says why.

In [ ]:
for name, reason in failed:
    print(f"{name:32s} {reason}")

Pablo's verdict on the same eight files, for comparison. Three of them Pablo
refuses too without extra input. One is DNA, which `Protein` does not claim.
The other four are two copies each of a cyclic peptide (`1csa`) and of a
peptide with a C-terminal `NH2` cap (`1hje`). Pablo handles both and
`Protein` does not yet.

In [ ]:
from openff.pablo import STD_CCD_CACHE, topology_from_pdb

STD_CCD_CACHE.auto_download = True
for name, _ in failed:
    try:
        topology = topology_from_pdb(cache / name)
        print(f"{name:32s} pablo loads it, {topology.n_atoms} atoms")
    except Exception as error:
        print(f"{name:32s} pablo refuses it too: {type(error).__name__}")